<a href="https://colab.research.google.com/github/sphilipose6/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"The total revenue across all 400 orders is ${total_revenue:,.2f}.")
print(f"The total number of units sold across all orders is {total_units}.")

The total revenue across all 400 orders is $8,520.00.
The total number of units sold across all orders is 783.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = df.groupby('category')[['revenue']].sum().sort_values('revenue', ascending=False)
by_category['share_pct'] = (by_category['revenue'] / total_revenue) * 100

by_category

,revenue,share_pct
category,,
Food,4293.0,50.387324
Merch,1771.5,20.792254
Drink,1554.0,18.239437
RainGear,901.5,10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
vendor_stats = df.groupby('vendor_id').agg(
    avg_revenue=('revenue', 'mean'),
    order_count=('revenue', 'count')
).sort_values('avg_revenue', ascending=False)

display(vendor_stats)

top_vendor = vendor_stats.index[0]
top_avg = vendor_stats.iloc[0]['avg_revenue']
print(f"Vendor {top_vendor} has the highest average order revenue at ${top_avg:.2f}, calculated across {vendor_stats.iloc[0]['order_count']} orders.")

,avg_revenue,order_count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


Vendor V-01 has the highest average order revenue at $22.60, calculated across 94.0 orders.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_revenue = df[df['category'] == 'Merch']['revenue'].sum()
merch_share = (merch_revenue / total_revenue) * 100

print(f"Merch accounts for {merch_share:.1f}% of the total revenue.")

Merch accounts for 20.8% of the total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
# Left join
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

# Prove row count and revenue total did not change
assert len(joined) == len(df), "Row count changed after merge!"
assert joined['revenue'].sum() == df['revenue'].sum(), "Total revenue changed after merge!"
print(f"Validation passed: Row count remains {len(joined)} and revenue remains ${joined['revenue'].sum():,.2f}.")

# Identify unmatched vendor
unmatched = joined[joined['vendor_name'].isna()]['vendor_id'].unique()[0]
print(f"The unmatched vendor is {unmatched}. I am filling the missing names with 'Unknown ({unmatched})'.")

# Handle unmatched vendor
joined['vendor_name'] = joined['vendor_name'].fillna(f'Unknown ({unmatched})')
joined.head()

Validation passed: Row count remains 400 and revenue remains $8,520.00.
The unmatched vendor is V-18. I am filling the missing names with 'Unknown (V-18)'.


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown (V-18)
2,V-18,Drink,3,4.5,13.5,Unknown (V-18)
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown (V-18)


**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot_report = pd.pivot_table(
    joined,
    values='revenue',
    index='vendor_name',
    columns='category',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
).fillna(0)

pivot_report

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown (V-18),582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

A. Based on the report, the vendors should prioritize maintaining high stock of food items, as Food drove the clear majority of sales, bringing in \$4,293.00. However, specific vendors like Hoos Burgers need to focus heavily on cross-selling; while they sold \$1,338.00 in food, they only managed \$171.00 in drink sales, suggesting they are missing out on easy beverage add-ons for their meals.

B. The least trustworthy answer is the vendor-specific breakdown in Q5 and Q6 due to the unmatched vendor, V-18. V-18 actually generated the most total revenue ($2,349.00) and had the highest number of orders (108), yet their identity is completely missing from the lookup table. Because the highest-grossing vendor is simply labeled "Unknown (V-18)," any business strategy or outreach aimed at our top seller is currently impossible until that missing data is tracked down.